#1.Problem Statement

Identify the Electric Vehicle Demand, based on the available data.

#2.Exploratory Data Analysis

##2.1 Loading a dataset

In [0]:
import pandas as pd

In [0]:
df = pd.read_csv("/Volumes/electric_vehicle/ev_sales/ev_sales_data_raw/train.csv")

##2.2 Inspecting the dataset

In [0]:
df.shape

In [0]:
df.info()

In [0]:
df["Daily_Commute_km"].describe()

###2.2.1 Checking for missing values

In [0]:
df.isna().sum()

###2.2.2 Slicing Data into groups

In [0]:
df.loc[0:10,["City_Type","Will_Buy_EV"]] #.loc screens by label

In [0]:
df.iloc[0:10, [9, 14]] #.iloc screens by index position

In [0]:
df.iloc[0:10, 9:14]

In [0]:
for idx, col in enumerate(df.columns):
    print(f"{idx}: {col}")

How many of these customers travel 30 km or more everyday?

In [0]:
thirty_km_commuters = df[df.Daily_Commute_km >= 30]

In [0]:
thirty_km_commuters.shape

#### 2.2.2.1 Reshaping into conditional tables

In [0]:
df.groupby("Will_Buy_EV")["Annual_Income_USD"].mean().round(2)

In [0]:
Will_buy_EV_pct = df["Will_Buy_EV"].value_counts(normalize=True)*100

In [0]:
Will_buy_EV_pct.round(2)

In [0]:
Annual_Income_EV_Yes = df.loc[df["Will_Buy_EV"] == "Yes","Annual_Income_USD"]

In [0]:
Annual_Income_EV_Yes.describe()

In [0]:
print(Annual_Income_EV_Yes.mean().round(2))

In [0]:
Annual_Income_EV_Yes.head()

In [0]:
Annual_Income_EV_Yes = Annual_Income_EV_Yes.to_frame(name="Income of Buyers")

In [0]:
Annual_Income_EV_Yes.head()

In [0]:
Annual_Income_EV_Yes = Annual_Income_EV_Yes.reset_index(drop=True)
Annual_Income_EV_Yes.head()

In [0]:
Annual_income_EV_No=df.loc[df["Will_Buy_EV"] == "No","Annual_Income_USD"]
Annual_income_EV_No.describe()

In [0]:
#grouping by two or more connditional columns - city type  and annnual income
Urban_income_group = df.loc[df["City_Type"] == "Urban",["Age","Annual_Income_USD","Daily_Commute_km","Will_Buy_EV"]]

In [0]:
Urban_income_group.shape

In [0]:

Urban_income_group["Will_Buy_EV"].value_counts(normalize=True)

In [0]:
df["City_Type"].unique()

**Create groups for suburban and rural customers.**

In [0]:
subrban_income_group = df.loc[df["City_Type"] == "Suburban",["Age","Annual_Income_USD","Will_Buy_EV"]]

In [0]:
rural_income_group = df.loc[df["City_Type"] == "Rural",["Age","Annual_Income_USD","Will_Buy_EV"]]

**Check the split of buyer preference in these groups**

In [0]:
Urban_income_group.value_counts("Will_Buy_EV")

In [0]:
Urban_income_group["Will_Buy_EV"].value_counts(normalize=True) * 100

In [0]:
Urban_income_group.describe()

In [0]:
Urban_income_group[Urban_income_group["Will_Buy_EV"] == "Yes"].describe()

In [0]:
Urban_income_group[Urban_income_group["Will_Buy_EV"] == "No"].describe()

**Observing Central Tendency across reshaped data**

In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].mean().round(2) #average

In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].median().round(2) #middle salary point

In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].std().round(2) #square root of the variance from the mean

In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].quantile(.25).round(2)

**Creating Custom dataframes and renaming columns by summary statistics**

In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].agg(['mean', 'count']).rename(columns={'mean': 'Avg_Annual_Income_USD', 'count': 'Count_People'})


In [0]:
Urban_income_group.groupby("Will_Buy_EV")["Annual_Income_USD"].agg(['mean', 'median', 'count', 'min', 'max']).rename(columns={'mean': 'Avg_Annual_Income_USD','median': 'Median_Annual_Income_USD','count': 'Count_People','min':'Min_Annual_Income_USD','max':'Max_Annual_Income_USD'})


In [0]:
# Split commuters into above and below 30 km
above_30 = df[df["Daily_Commute_km"] > 30]
below_30 = df[df["Daily_Commute_km"] <= 30]

above_30_pct = (above_30["Will_Buy_EV"].value_counts(normalize=True) * 100).round(2)
below_30_pct = (below_30["Will_Buy_EV"].value_counts(normalize=True) * 100).round(2)

commute_buy_preference = pd.DataFrame({
    "Commute > 30 km (%)": above_30_pct,
    "Commute <= 30 km (%)": below_30_pct,
})


commute_buy_preference

###2.2.3 Visualizing Distributions on unscaled data for business reporting 

In [0]:
df["Will_Buy_EV"].unique()

In [0]:
df["Will_Buy_EV"].isna().sum()

In [0]:
df["y"] = df["Will_Buy_EV"].map({"Yes": 1, "No": 0}) #Convert to integer datatype

In [0]:
print(df["Will_Buy_EV"].unique())            # check the exact labels first



print(df["y"].isna().sum())                  # must be 0
print(df["y"].value_counts())

In [0]:
pip install seaborn

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# Select only numerical columns and compute the correlation matrix
numerical_df = df.select_dtypes(include="number")

correlation_matrix = numerical_df.corr()

correlation_matrix

In [0]:

plt.figure(figsize=(10, 8))

sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f", square=True, linewidths=0.5)

plt.title("Correlation Heatmap")

plt.tight_layout()

plt.show()

### 2.2.4 Transforming the data from qualitative to quantitative data types


In [0]:
df.info()

In [0]:
df["Gender"].value_counts() #inspecting the need for dummy variables

In [0]:
gender_dummies = pd.get_dummies(df["Gender"], prefix="Gender", dtype=int) #Creating the transformation of the gender column

In [0]:
new_cols = gender_dummies.loc[:, ~gender_dummies.columns.isin(df.columns)] #only dummy columns not already present in df
if not new_cols.empty:
    df = pd.concat([df, new_cols], axis=1) #Adding the dummy variables to the dataframe (safe to re-run)

In [0]:
df = df.drop(["Gender_Female","Gender_Male","Gender_Other"], axis=1)

In [0]:
df[["Gender_Female","Gender_Male","Gender_Other","y"]].corr()["y"]

In [0]:
df["City_Type"].value_counts()


In [0]:
city_type_dummies = pd.get_dummies(df["City_Type"], prefix="City_Type", dtype=int)



In [0]:

new_cols = city_type_dummies.loc[:, ~city_type_dummies.columns.isin(df.columns)] #only dummy columns not already present in df
if not new_cols.empty:
    df = pd.concat([df, new_cols], axis=1) #Adding the dummy variables to the dataframe (safe to re-run)

In [0]:
df[["City_Type_Urban", "City_Type_Suburban", "City_Type_Rural", "y"]].corr()["y"]

In [0]:
df["Current_Car_Type"].value_counts()


In [0]:
current_car_type_dummies = pd.get_dummies(df["Current_Car_Type"], prefix="Current_Car_Type", dtype=int)

In [0]:
new_cols = current_car_type_dummies.loc[:, ~current_car_type_dummies.columns.isin(df.columns)] #only dummy columns not already present in df
if not new_cols.empty:
    df = pd.concat([df, new_cols], axis=1) #Adding the dummy variables to the dataframe (safe to re-run)

In [0]:
df[["Current_Car_Type_Hatchback", "Current_Car_Type_SUV", "Current_Car_Type_Sedan", "Current_Car_Type_Truck","y"]].corr()["y"]

#### 2.2.4.1 Creating a function to automate transformation and parsing of the one-hot encoded/dummy variables to the dataframe

In [0]:
def add_dummies_to_df(df, columns):
    """Create dummy variables for the given columns and append them to df (skips columns already dummified, safe to re-run)."""
    for col in columns:
        if col not in df.columns:
            print(f"Column '{col}' not found in df, skipping.")
            continue
        dummies = pd.get_dummies(df[col], prefix=col, dtype=int)
        new_cols = dummies.loc[:, ~dummies.columns.isin(df.columns)]  # only dummy columns not already present
        if not new_cols.empty:
            df = pd.concat([df, new_cols], axis=1)
    return df



In [0]:
help(add_dummies_to_df) #reading the docstring we created

In [0]:
df = add_dummies_to_df(df, ["City_Type","Current_Car_Type","Gender"])



In [0]:
df.head()

In [0]:
# Select only numerical columns and compute the correlation matrix
numerical_df = df.select_dtypes(include="number")

correlation_matrix = numerical_df.corr()

correlation_matrix

In [0]:
df["Range_Anxiety_Level"].unique()

In [0]:
#mapping categorical variables to numerical values in the same column
df["Range_Anxiety_Level_Converted_Quant"] = df["Range_Anxiety_Level"].map({"Low": 0, "Medium": 1, "High": 2})

In [0]:
df["Range_Anxiety_Level_Converted_Quant"].unique()

In [0]:
df = df.drop(["Range_Anxiety_Level_High", "Range_Anxiety_Level_Medium", "Range_Anxiety_Level_Low"], axis=1)

In [0]:
df["Subsidy_Available"].unique()

In [0]:
df["Subsiby_Available_Converted_Quant"] = df["Subsidy_Available"].map({"No": 0, "Yes": 1})


In [0]:
df["Home_Charging_Possible"].unique()


In [0]:
df["Home_Charging_Possible_Converted_Quant"] = df["Home_Charging_Possible"].map({"No": 0, "Yes": 1})
                                                                                 


In [0]:
pip install seaborn

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns


numerical_df = df.select_dtypes(include="number")

correlation_matrix = numerical_df.corr()

correlation_matrix

plt.figure(figsize=(10, 8))

sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f", square=True, linewidths=0.5)

plt.title("Correlation Heatmap")

plt.tight_layout()

plt.show()

In [0]:
import numpy as np

In [0]:
numerical_df.shape

In [0]:
pip install seaborn

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [0]:
plt.figure(figsize=(14, 11))

# Mask the upper triangle so each pair is shown only once (less visual clutter)
# Drop the id column — it carries no meaningful correlation information
corr_plot = correlation_matrix.drop(index="id", columns="id", errors="ignore")

mask = np.triu(np.ones_like(corr_plot, dtype=bool))

sns.heatmap(
    corr_plot,
    mask=mask,
    annot=True,
    annot_kws={"size": 8},
    cmap="coolwarm",
    center=0,
    fmt=".2f",
    square=True,
    linewidths=0.8,
    linecolor="white",
    cbar_kws={"shrink": 0.8, "label": "Correlation"},
)

plt.title("Correlation Heatmap (lower triangle)", fontsize=14)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(fontsize=9)

plt.tight_layout()

plt.show()

##2.3 Hypothesis Framing

## A. Main effects: what drives Will_Buy_EV

###2.3.1 H1: Buyers have higher environmental concern than non-buyers

In [0]:
# Box plot of Environmental Concern for buyers (y=1) vs non-buyers (y=0)
plt.figure(figsize=(8, 6))

sns.boxplot(
    data=df,
    x="y",
    y="Environmental_Concern_Level",
    hue="y",
    palette={0: "#1f77b4", 1: "#d62728"},
    legend=False,
)

plt.title("Environmental Concern: EV Buyers vs Non-Buyers", fontsize=14)
plt.xlabel("Bought EV")
plt.ylabel("Environmental Concern")
plt.xticks(ticks=[0, 1], labels=["Non-Buyers (0)", "Buyers (1)"])

plt.tight_layout()
plt.show()


 Reading the Box Plot: Quantiles Explained & Inference


**What each part of the box plot represents (quantile labels):**

A box plot summarizes the distribution of **Environmental Concern** for each group using five key numbers (the "five-number summary"):

| Element | Quantile it represents | Interpretation |
|---|---|---|
| **Bottom whisker** | Roughly Q1 − 1.5×IQR (or the minimum non-outlier) | The lowest concern scores within 1.5×IQR of the box; values beyond are plotted as outlier dots |
| **Bottom of the box** | **Q1 (25th percentile)** | 25% of respondents in this group score at or below this concern level |
| **Line inside the box** | **Median (Q2, 50th percentile)** | The "typical" respondent — half score below, half above; robust to skew/outliers, unlike the mean |
| **Top of the box** | **Q3 (75th percentile)** | 75% of respondents score at or below this level |
| **Top whisker** | Roughly Q3 + 1.5×IQR (or the maximum non-outlier) | The highest concern scores that are still not classified as outliers |
| **Box height (Q3 − Q1)** | **IQR (Interquartile Range)** | The spread of the middle 50% of the data — how much concern varies among the central mass of respondents |
| **Dots beyond whiskers** | Outliers | Individual responses with unusually low/high concern relative to their group |

**Inference from the plot:**

1. **Location shift:** The buyers' box (y=1) sits noticeably higher on the concern axis than the non-buyers' box — the buyers' median line is close to where the non-buyers' Q3 lies. This suggests the *typical* buyer reports substantially higher environmental concern than the *typical* non-buyer.
2. **Limited overlap:** Because the boxes overlap only slightly, the two distributions are largely separated, hinting at a strong effect size (later confirmed by Cohen's d) rather than a marginal difference.
3. **Spread (IQR):** Comparing box heights shows whether concern varies more in one group — e.g., if non-buyers show a taller box, their concern is more heterogeneous, meaning "non-buyer" pools together low- and medium-concern people.
4. **Outliers:** Any dots beyond the whiskers flag unusual respondents (e.g., low-concern buyers or high-concern non-buyers). These are the interesting "counter-intuitive" cases worth profiling — high-concern non-buyers likely face a practical barrier (cost, charging, range) that the t-test cannot detect.
5. **Shape of distribution:** If the median line sits closer to one end of a box, that group's distribution is skewed (e.g., buyers' median near the top of their box would indicate most buyers cluster at high concern).

**Conclusion (visual, pre-test):** The box plot supports H1 visually — buyers report systematically higher environmental concern than non-buyers. This observation motivates the formal Welch's t-test, Cohen's d effect size, and Mann-Whitney U robustness check that follow.

In [0]:
# Histogram (100 bins) of Environmental Concern for buyers (y=1) vs non-buyers (y=0)
# to check the shape/normality of the distributions
plt.figure(figsize=(12, 6))

sns.histplot(
    data=df,
    x="Environmental_Concern_Level",
    hue="y",
    bins=30,
    element="step",       # outline style keeps overlapping bins readable
    stat="density",       # density makes the two groups comparable despite different sizes
    common_norm=False,    # normalize each group separately
    palette={0: "#1f77b4", 1: "#d62728"},
)

plt.title("Environmental Concern Distribution: EV Buyers vs Non-Buyers (100 bins)", fontsize=14)
plt.xlabel("Environmental Concern")
plt.ylabel("Density")
plt.legend(title="Bought EV", labels=["Non-Buyers (0)", "Buyers (1)"])

plt.tight_layout()
plt.show()


####2.3.1.1 Welch t-test for groups have inequal variances or inequal sample sizes

In [0]:
import numpy as np                                     
from scipy import stats

# Split Environmental Concern into the two groups: buyers (y=1) and non-buyers (y=0)
concern_buyers = df.loc[df["y"] == 1, "Environmental_Concern_Level"].dropna()
concern_non_buyers = df.loc[df["y"] == 0, "Environmental_Concern_Level"].dropna()




In [0]:
alpha = 0.05
conf_level = 1 - alpha                                 # 95% confidence



In [0]:
# Basic group statistics                                
n1, n2 = len(concern_buyers), len(concern_non_buyers)
m1, m2 = concern_buyers.mean(), concern_non_buyers.mean()
s1, s2 = concern_buyers.std(), concern_non_buyers.std()

In [0]:
# Welch's t-test (does NOT assume equal variances or equal sample sizes)
t_stat, p_value = stats.ttest_ind(concern_buyers, concern_non_buyers, equal_var=False)

In [0]:
print(f"Level of significance is {p_value:.5f}")

In [0]:

# 95% CI for the DIFFERENCE in means (Welch method)
diff = m1 - m2
var_m1, var_m2 = s1**2 / n1, s2**2 / n2                # variance of each sample mean
se_diff = np.sqrt(var_m1 + var_m2)                     # standard error of the difference
dof = (var_m1 + var_m2)**2 / (var_m1**2 / (n1 - 1) + var_m2**2 / (n2 - 1))  # Welch–Satterthwaite df
t_crit = stats.t.ppf(1 - alpha / 2, dof)               # critical t value
ci_low, ci_high = diff - t_crit * se_diff, diff + t_crit * se_diff

In [0]:
# 95% CI for each group's own mean (per-group SE and t critical value)
se1, se2 = s1 / np.sqrt(n1), s2 / np.sqrt(n2)
tc1, tc2 = stats.t.ppf(1 - alpha / 2, n1 - 1), stats.t.ppf(1 - alpha / 2, n2 - 1)
b_low, b_high = m1 - tc1 * se1, m1 + tc1 * se1
nb_low, nb_high = m2 - tc2 * se2, m2 + tc2 * se2


In [0]:
# Output
print(f"Buyers (y=1):     n={n1:,}, mean={m1:.3f}, std={s1:.3f}, 95% CI [{b_low:.3f}, {b_high:.3f}]")
print(f"Non-buyers (y=0): n={n2:,}, mean={m2:.3f}, std={s2:.3f}, 95% CI [{nb_low:.3f}, {nb_high:.3f}]")
print(f"\nMean difference (buyers − non-buyers): {diff:.3f}")
print(f"95% CI for the difference:            [{ci_low:.3f}, {ci_high:.3f}]")
print(f"Relative difference:                  {diff / m2 * 100:.1f}% higher than non-buyers")
print(f"\nWelch's t-statistic: {t_stat:.4f}  (df = {dof:,.0f})")
print(f"p-value: {p_value:.2e}")                       # CHANGED: scientific notation

In [0]:
if p_value < alpha:
    print(f"\n p < {alpha}: Significant difference in Environmental Concern between buyers and non-buyers.")
    print(f"The true difference is plausibly between {ci_low:.3f} and {ci_high:.3f} points.")
else:
    print(f"\n p >= {alpha}: No significant difference in Environmental Concern between buyers and non-buyers.")
    print(f"The CI [{ci_low:.3f}, {ci_high:.3f}] includes 0, so 'no difference' cannot be ruled out.")

#### 2.3.1.2 Cohen's d Effect Size of Environmental Concern on Buyer behaviour

In [0]:

n1, n0 = len(concern_buyers), len(concern_non_buyers)
mean1, mean0 = concern_buyers.mean(), concern_non_buyers.mean()
var1, var0 = concern_buyers.var(ddof=1), concern_non_buyers.var(ddof=1)

# Cohen's d uses the pooled standard deviation (weighted by sample size)
pooled_sd = np.sqrt(((n1 - 1) * var1 + (n0 - 1) * var0) / (n1 + n0 - 2))
cohens_d = (mean1 - mean0) / pooled_sd

print(f"Buyers (y=1):     n={n1}, mean={mean1:.3f}, std={concern_buyers.std():.3f}")
print(f"Non-buyers (y=0): n={n0}, mean={mean0:.3f}, std={concern_non_buyers.std():.3f}")
print(f"Pooled SD: {pooled_sd:.3f}")
print(f"Cohen's d: {cohens_d:.3f}")

# Interpretation using Cohen's conventional thresholds
abs_d = abs(cohens_d)
if abs_d < 0.2:
    interpretation = "negligible"
elif abs_d < 0.5:
    interpretation = "small"
elif abs_d < 0.8:
    interpretation = "medium"
else:
    interpretation = "large"

print(f"Effect size interpretation: {interpretation} effect")

# Optional: 95% CI for Cohen's d via the standard error approximation (Hedges & Olkin)
se_d = np.sqrt((n1 + n0) / (n1 * n0) + cohens_d**2 / (2 * (n1 + n0)))
ci_low, ci_high = cohens_d - 1.96 * se_d, cohens_d + 1.96 * se_d
print(f"Approximate 95% CI for Cohen's d: [{ci_low:.3f}, {ci_high:.3f}]")


####2.3.1.3 Mann-Whitney U test for Robustness Check

In [0]:
from scipy import stats

# Mann-Whitney U test (non-parametric) as a robustness check for H1:
# does Environmental Concern still differ between buyers and non-buyers without assuming normality or equal variances?

# Medians for context (the Mann-Whitney U test compares distributions/ranks)
print(f"Buyers (y=1): n={len(concern_buyers)}, median={concern_buyers.median():.3f}, mean rank summary")

print(f"Non-buyers (y=0): n={len(concern_non_buyers)}, median={concern_non_buyers.median():.3f}")

# Mann-Whitney U test (two-sided)
u_stat, p_value = stats.mannwhitneyu(concern_buyers, concern_non_buyers, alternative="two-sided")

# Rank-biserial correlation as the effect size for Mann-Whitney
rank_biserial = ((2 * u_stat)- 1)/ (n1 * n0)  # = (2U/(n1*n0)) - 1 with U for group 1
print(f"\nMann-Whitney U statistic: {u_stat:.1f}")
print(f"p-value: {p_value:.6f}")
print(f"Rank-biserial correlation (effect size): {rank_biserial:.3f}")

alpha = 0.05
if p_value < alpha:
    print(f"\np < {alpha}: The non-parametric test confirms a significant difference in Environmental "
          "Concern between buyers and non-buyers (robust to non-normality).")
else:
    print(f"\np >= {alpha}: The non-parametric test finds no significant difference in Environmental "
          "Concern between buyers and non-buyers.")


####2.3.1.4 DECISION ON H1: "ENVIRONMENTAL CONCERN HAS A SIGNIFICANT IMPACT ON BUYER BEHAVIOUR"

**Storyline for H1:** Environmental concern is the Single strong signal of EV purchase intent. Buyers score 1.75 points higher on environmental concern (4.38 vs 2.63), 66% higher than non-buyers. The estimate is highly precise (95% CI: 1.74–1.75).

BUSINESS IMPLICATIONS
- **Targeting:** 
Prioritise high-concern customers in campaigns. They are the most receptive audience.
- **Conversion:** for high-concern non-buyers, the problem is not the message. Find their barrier and remove it: subsidy information, home-charging installation, or range reassurance.
- **Messaging by segment:** lead with environmental benefits for high-concern customers. For low-concern customers, lead with cost savings and subsidies, which are your second-strongest driver by the correlation yet to be confirmed.

###2.3.2 H2: The Yes rate is higher when a subsidy is available

####2.3.2.1 ChiSquare Test for establishing H2

In [0]:
import pandas as pd
from scipy import stats

# H2: Is the Yes rate (y = bought EV) associated with subsidy availability?
# Chi-square test of independence on the contingency table of Subsidy_Available x y

# Contingency table: rows = subsidy status, columns = buy decision (y)
contingency = pd.crosstab(df["Subsidy_Available"], df["y"])
print("Observed counts (rows = subsidy availability, columns = y):")
print(contingency)



In [0]:
# Chi-square test of independence
chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency)

alpha = 0.05
print(f"\nChi-square statistic: {chi2_stat:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value:.4g}")


In [0]:
# Check expected counts (chi-square assumes all expected frequencies >= 5)
if (expected < 5).any():
    print("\nWARNING: some expected counts are below 5 — consider Fisher's exact test instead.")
else:
    print("All expected counts are >= 5 (chi-square assumption met).")


In [0]:
# Yes rates per subsidy group for direction of the effect
crosstab_norm = pd.crosstab(df["Subsidy_Available"], df["y"], normalize="index")
print("\nRow-wise Yes rate (y=1) per subsidy status:")
print(crosstab_norm)

if p_value < alpha:
    print(f"\np < {alpha}: The Yes rate differs significantly by subsidy availability — H2 supported.")
else:
    print(f"\np >= {alpha}: No significant association between subsidy availability and the Yes rate.")

### 2.3.3 H3: The Yes rate differs across range-anxiety levels (Low > Medium > High)

### 2.3.4 H4: The Yes rate is higher when home charging is possible

### 2.3.5 H5: Buyers have higher annual income

### 2.3.6 H6: Buyers have shorter daily commutes

## B. “No effect” hypotheses: variables near zero

### 2.3.7 H7: Gender has no meaningful effect on the Yes rate

### 2.3.8 H8: Age has no meaningful effect on the Yes rate

###2.3.9 H9: City type has no meaningful direct effect on the Yes rate

##C. Relationships among the predictors

### 2.3.10 H10: Urban residents are less likely to have home charging than rural or suburban residents

### 2.3.11 H11: Urban residents have more public charging stations near home and work

### 2.3.12 H12: People with home charging are more likely to report low range anxiety

### 2.3.13 H13: People with longer commutes report higher range anxiety

## D. Interaction and mediation hypotheses

### 2.3.14 H14: Subsidies raise the Yes rate more for lower-income people

### 2.3.15 H15: Environmental concern turns into buying mainly when income is high enough

### 2.3.16 H16: A long commute lowers the Yes rate less when home charging is possible

### 2.3.17 H17: Home charging raises the Yes rate through lower range anxiety

### 2.3.18 H18: Environmental concern, subsidy and income stay significant after controlling for all other variables

#3.Machine Learning - Logistic Regression Model 

##3.1 Importing the required libraries

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from mlflow.models import infer_signature

mlflow.set_experiment("/Users/doctornikhildjonathan@gmail.com/ev_numerical_logistic_regression")

In [0]:
X = numerical_df.drop(columns=["y"])
y = numerical_df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]} rows, Test set: {X_test.shape[0]} rows")
print(f"Yes rate - train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")

In [0]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

with mlflow.start_run(run_name="logistic_regression_numerical_df") as run:
    import time
    start = time.time()
    pipeline.fit(X_train, y_train)
    train_time = time.time() - start

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "train_time_sec": train_time,
    }
    mlflow.log_param("features", list(X.columns))
    mlflow.log_param("n_train", X_train.shape[0])
    mlflow.log_param("n_test", X_test.shape[0])
    mlflow.log_metric("n_final_fit", numerical_df.shape[0])
    mlflow.log_metrics(metrics)

    signature_input = X_train.head(100)
    signature = infer_signature(signature_input, pipeline.predict(signature_input))

    model_info = mlflow.sklearn.log_model(
        pipeline,
        name="logistic_regression_model",
        signature=signature,
        input_example=X_train.head(3),
    )

print("Metrics:")
for name, value in metrics.items():
    print(f"  {name}: {value:.4f}")
print(f"\nMLflow run id: {run.info.run_id}")
print(f"Model URI: {model_info.model_uri}")

In [0]:
mlflow.register_model(model_uri="models:/m-44e57f720bda4c0e849cef2204935256", name="electric_vehicle.ev_sales.ev_demand_classifier")

In [0]:
import time
from mlflow.models import infer_signature

# Variants of the logistic regression model to compare as child runs
variants = [
    {"name": "baseline",            "C": 1.0,  "solver": "lbfgs",     "penalty": "l2", "class_weight": None},
    {"name": "strong_reg_c0_01",    "C": 0.01, "solver": "lbfgs",     "penalty": "l2", "class_weight": None},
    {"name": "weak_reg_c10",        "C": 10.0, "solver": "lbfgs",     "penalty": "l2", "class_weight": None},
    {"name": "balanced_weights",   "C": 1.0,  "solver": "lbfgs",     "penalty": "l2", "class_weight": "balanced"},
    {"name": "liblinear_l1",       "C": 1.0,  "solver": "liblinear", "penalty": "l1", "class_weight": None},
]

results = []
best = None

with mlflow.start_run(run_name="EV_Demand_Classifier_LogisticRegression_parent") as parent_run:
    mlflow.log_param("model_family", "logistic_regression")
    mlflow.log_param("n_variants", len(variants))
    mlflow.log_param("n_train", X_train.shape[0])
    mlflow.log_param("n_test", X_test.shape[0])
    mlflow.log_param("features", list(X.columns))

    for var in variants:
        with mlflow.start_run(run_name=f"variant_{var['name']}", nested=True) as child_run:
            pipeline = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(
                    C=var["C"],
                    solver=var["solver"],
                    penalty=var["penalty"],
                    class_weight=var["class_weight"],
                    max_iter=1000,
                    random_state=42,
                )),
            ])

            start = time.time()
            pipeline.fit(X_train, y_train)
            train_time = time.time() - start

            y_pred = pipeline.predict(X_test)
            y_prob = pipeline.predict_proba(X_test)[:, 1]

            metrics = {
                "accuracy": accuracy_score(y_test, y_pred),
                "precision": precision_score(y_test, y_pred),
                "recall": recall_score(y_test, y_pred),
                "f1": f1_score(y_test, y_pred),
                "roc_auc": roc_auc_score(y_test, y_prob),
                "train_time_sec": train_time,
            }

            mlflow.log_params({k: str(v) for k, v in var.items() if k != "name"})
            mlflow.log_metrics(metrics)

            signature_input = X_train.head(100)
            signature = infer_signature(signature_input, pipeline.predict(signature_input))
            model_info = mlflow.sklearn.log_model(
                pipeline,
                name=f"logreg_{var['name']}",
                signature=signature,
                input_example=X_train.head(3),
            )

            results.append({
                "variant": var["name"],
                "run_id": child_run.info.run_id,
                "model_uri": model_info.model_uri,
                "pipeline": pipeline,
                "metrics": metrics,
            })

            if best is None or metrics["roc_auc"] > best["metrics"]["roc_auc"]:
                best = results[-1]

    # Log the best variant's params/metrics on the parent run
    mlflow.log_params({f"best_{k}": str(v) for k, v in variants(results and None or []).items()} if False else {f"best_{k}": str(v) for k, v in [("name", best["variant"]), ("C", [v for v in variants if v["name"] == best["variant"]][0]["C"]), ("solver", [v for v in variants if v["name"] == best["variant"]][0]["solver"]), ("penalty", [v for v in variants if v["name"] == best["variant"]][0]["penalty"]), ("class_weight", [v for v in variants if v["name"] == best["variant"]][0]["class_weight"])]})
    mlflow.log_metrics({f"best_{k}": v for k, v in best["metrics"].items()})

    # Log the winning fitted pipeline as the parent run's model
    signature_input = X_train.head(100)
    signature = infer_signature(signature_input, best["pipeline"].predict(signature_input))
    best_model_info = mlflow.sklearn.log_model(
        best["pipeline"],
        name="EV_Demand_Classifier_LogisticRegression",
        signature=signature,
        input_example=X_train.head(3),
    )

print(f"Parent run id: {parent_run.info.run_id}")
print("\nVariant comparison (test set):")
print(f"{'variant':<20} {'accuracy':>9} {'precision':>10} {'recall':>8} {'f1':>8} {'roc_auc':>8}")
for r in results:
    m = r["metrics"]
    print(f"{r['variant']:<20} {m['accuracy']:>9.4f} {m['precision']:>10.4f} {m['recall']:>8.4f} {m['f1']:>8.4f} {m['roc_auc']:>8.4f}")
print(f"\nBest variant: {best['variant']} (roc_auc={best['metrics']['roc_auc']:.4f})")

# Register the best model in Unity Catalog
mlflow.set_registry_uri("databricks-uc")
registered_model_name = "electric_vehicle.ev_sales.EV_Demand_Classifier_LogisticRegression"
registered_version = mlflow.register_model(
    model_uri=best_model_info.model_uri,
    name=registered_model_name,
    await_registration_for=300,
)
print(f"\nRegistered model: {registered_model_name}, version: {registered_version.version}")
print(f"Model URI: {best_model_info.model_uri}")